In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.applications import mobilenet_v2

# 하이퍼파라미터 설정
BATCH_SIZE = 32
IMG_SIZE = (224,224)
DATA_DIR = '../data/dataset_imagenet'
[1,2,3] # 리스트 
{1,2,3} # set
{'name' : 'hong'} # 딕셔너리
(1,2) # 튜플

# 이미지 데이터 생성
# 학습 데이터
train_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='training',
	seed = 1000,
	image_size = IMG_SIZE,
	batch_size = BATCH_SIZE
)
# 검증 데이터
val_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='validation',
	seed = 1000,
	image_size = IMG_SIZE,
	batch_size = BATCH_SIZE
)

Found 20 files belonging to 2 classes.
Using 16 files for training.
Found 20 files belonging to 2 classes.
Using 4 files for validation.


In [3]:
train_ds

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [4]:
# 성능 최적화
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(20).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# 모델 설계
# 전이학습 모델 가져옴
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False, weights='imagenet')

# 데이터 증강 => 이미지를 돌리고 확대하고 움직여서 여러 이미지를 추가하는 작업
data_augmentation = tf.keras.Sequential([
	layers.RandomFlip('horizontal_and_vertical'),
	layers.RandomRotation(0.2),
	layers.RandomZoom(0.2)
])

# 새 모델 설계
model = models.Sequential([
	layers.Input((224,224,3)),
	data_augmentation,
	# 정규화를 0~1이 아닌 -1~1로 해야 함. 왜? imagenet에서 그렇게 했기 때문에 맞춰해야함
	layers.Rescaling(1./127.5, offset=-1),
	base_model,
	layers.GlobalAveragePooling2D(),
	layers.Dense(128, activation='relu'),
	layers.Dropout(0.2),
	layers.Dense(2, activation='softmax')
])

C:\Users\C603\AppData\Local\Temp\ipykernel_5100\4045717380.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=(244,244,3),include_top=False, weights='imagenet')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# 모델 설정
model.compile(optimizer='adam', loss='spar')
history = model
history = model.fit(X, y, verbose=0, epochs=100, shuffle=True, validation_split=0.2)

In [ ]:
from tensorflow.keras.pre import mobilenet_v2
from tensorflow.keras.applications import mobilenet_v2
import requests
from io import BytesIO
import numpy as np
def predict_img_url(url):
	# 모델 불러오기. 분류기 포함 (1000개의 분류)
	model = mobilenet_v2.MobileNetV2(weights='imagenet', include_top=True)
	# 이미지 정보 가져오기
	response = requests.get(url)
	# 가져온 이비지를 바이트형태로 변환
	img = image.load_img(BytesIO(response.content), target_size=(224,224))

	# 이미지 전처리 => 입력 형태 맞추기, 이미지를 배열로
	X = image.img_to_array(img)
	X = np.expand_dims(X, axis=0)

	# MobileNetV2 전용 전처리
	X = mobilenet_v2.preprocess_input(X)

#	예특
	predicts = model.predict(X)

	# 4. 결과 해석 (중요: 함수 내부의 예측값인 'predicts'를 사용해야 함)
  # decode_predictions를 사용하여 상위 3개를 가져옵니다.
	current_results = mobilenet_v2.decode_predictions(predicts, top=3)[0]
	print('-'*30)
	for i, (_, label, prob) in enumerate(current_results):
		print(f'{i+1}. {label}:{prob*100:.2f}%')
	print('-'*30)
	